linear regression full pipeline.
predicting house prices from sq footage,bedrooms, and bathrooms
1. fit the model
2. look at coeff.
3. compute mse/rmse
4. compute r2
5. get a p value per coeff.(t test)
6. get f test for whole model.

In [4]:
!pip install statsmodels


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import  numpy as np #(np) does number crunching, random numbers, arrays, math.
import pandas as pd #(pd) handles tables of data, like Excel but in code..
from sklearn.linear_model import LinearRegression #actual linear regression tool, it does the fitting for you.
from sklearn.metrics import mean_squared_error, r2_score #ready-made functions so you don't calculate MSE or R² by hand.
from sklearn.model_selection import train_test_split #splits your data into a training chunk and a testing chunk.
import statsmodels.api as sm
# statsmodels is second library, used only because sklearn doesn't give you p-values or
#  F-tests. sklearn is built for prediction, statsmodels is built for statistical explanation.


In [6]:
#fake dataset
np.random.seed(42) # locks randomness for each runwithout shuffling.
n=200 # 200 fake houses
sqft= np.random.normal(1500,400,n) # 200 random house around 1500 sqft with 400 std
bedrooms = np.random.randint(1,5,n) # randint gives whole num.
bathrooms = np.random.randint (1,4,n)
commute_dis =np.random.normal(10,5,n) # deliberately WEAK / irrelevant feature
#true price relationship : price depends on sqft,bedrooms, bathrooms, but not on commute_dis.it has no real effect,(coeff 0) - pure noise added
noise = np.random.normal(0,20000,n)#normal gives decimal-ish spread values.
price = (50*sqft )+(8000*bedrooms)+(12000*bathrooms)+50000+noise

In [7]:
# df stands for DataFrame, a table. (pd) 
df = pd.DataFrame({ #packages all those separate number lists into 
    #one neat table, like columns in a spreadsheet
    "sqft": sqft,
    "bedrooms":bedrooms,
    "bathrooms":bathrooms,
    "commute_dis": commute_dis,
    "price":price
})

In [11]:
print("="*70)
print("STEP 0: THE DATA")
print("="*70)
print(df.head())
print(f"\n{n} rows total\n")
 
X = df[["sqft", "bedrooms", "bathrooms", "commute_dis"]] #input features (independent variables)
y = df["price"] #target variable (dependent variable)
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #keeps the split identical every run.) 
 
# ------------------------------------------------------------------
# 1. FIT THE MODEL (sklearn) — this is least squares minimizing MSE
# ------------------------------------------------------------------
print("="*70)
print("STEP 1: FIT THE MODEL (least squares -> minimizes MSE)")
print("="*70)
 
model = LinearRegression() #creates an empty, unfit linear regression object, think of it as an empty recipe card.
model.fit(X_train, y_train)#actual learning step, this is where least squares runs behind the
# scenes and finds the b0, b1, b2, b3 that minimize MSE on the training data.
 
print(f"Intercept (b0): {model.intercept_:.2f}")
for feature, coef in zip(X.columns, model.coef_):
    print(f"Coefficient for {feature}: {coef:.2f}")
 #After fitting, sklearn stores the answers inside the model object.
 #  intercept_ is b0. coef_ is a list of b1, b2, b3, one per feature, in the
 #  same order as your X columns. The trailing underscore is sklearn's convention 
 # for "this was learned, not something you set

# ------------------------------------------------------------------
# 2. PREDICTIONS + MSE / RMSE
# ------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 2: MSE / RMSE — how wrong is the model, on average?")
print("="*70)
 
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions) #ompares your predictions against the real known prices and computes that average-squared-error number
rmse = np.sqrt(mse)
 
print(f"MSE  : {mse:,.2f}   <- squared rupees, not interpretable directly")
print(f"RMSE : {rmse:,.2f}   <- back in rupees, e.g. 'off by about ₹{rmse:,.0f} on average'")
 
# ------------------------------------------------------------------
# 3. R² — how much variance does the model explain overall?
# ------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 3: R² — fraction of variance in price explained by the model")
print("="*70)
 
r2 = r2_score(y_test, predictions)
print(f"R² : {r2:.3f}  ->  model explains {r2*100:.1f}% of the variation in price")
print(f"     the remaining {(1-r2)*100:.1f}% is unexplained (noise, missing features)")
 
# ------------------------------------------------------------------
# 4. P-VALUES per coefficient + F-TEST for the whole model
#    sklearn does NOT give you these -> need statsmodels
# ------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 4: P-VALUES (per coefficient) + F-TEST (whole model)")
print("        [sklearn can't do this — statsmodels can]")
print("="*70)
 
X_train_sm = sm.add_constant(X_train)  # statsmodels needs the intercept added manually
sm_model = sm.OLS(y_train, X_train_sm).fit()
 
print(sm_model.summary())
 
# ------------------------------------------------------------------
# 5. HOW TO READ THE SUMMARY ABOVE (this is the part interviews test)
# ------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 5: READING THE OUTPUT LIKE AN INTERVIEWER WOULD")
print("="*70)
 
print("""
Look at the 'P>|t|' column in the summary table above:
 
  - sqft, bedrooms, bathrooms  -> p-value should be tiny (<< 0.05)
    -> reject null hypothesis for these -> real relationship, trust the coefficient
 
  - commute_distance -> p-value should be LARGE (likely > 0.05)
    -> fail to reject null -> this coefficient is probably just noise
    -> we built the data this way on purpose, commute_distance has zero
       true relationship to price, this is what a "fake but convincing"
       feature looks like in a real interview dataset
 
Now look at 'Prob (F-statistic)' near the top of the summary:
 
  - this tests the WHOLE model at once: "does at least one feature matter?"
  - it should be extremely small (something like 0.000)
  - -> reject the null -> the model overall is clearly better than
       just predicting the average price every time
 
This is the exact chain from theory to numbers:
  fit -> coefficients -> MSE/RMSE -> R² -> per-feature p-values -> F-test
""")

STEP 0: THE DATA
          sqft  bedrooms  bathrooms  commute_dis          price
0  1698.685661         2          1    11.708780  153747.065069
1  1444.694280         2          3    19.380854  157237.826587
2  1759.075415         1          1    14.752119  174560.487093
3  2109.211943         4          1     7.115482  182338.920610
4  1406.338650         1          2     5.507927  153748.257250

200 rows total

STEP 1: FIT THE MODEL (least squares -> minimizes MSE)
Intercept (b0): 50855.77
Coefficient for sqft: 43.27
Coefficient for bedrooms: 9656.15
Coefficient for bathrooms: 12219.43
Coefficient for commute_dis: 445.55

STEP 2: MSE / RMSE — how wrong is the model, on average?
MSE  : 486,438,826.39   <- squared rupees, not interpretable directly
RMSE : 22,055.36   <- back in rupees, e.g. 'off by about ₹22,055 on average'

STEP 3: R² — fraction of variance in price explained by the model
R² : 0.256  ->  model explains 25.6% of the variation in price
     the remaining 74.4% is unexp

**LOGISTIC REGRESSTION** IN SKLEARN

In [ ]:
from sklearn.linear_model import LogisticRegression #brings in the LogisticRegression class.
from sklearn.model_selection import train_test_split #brings in a function that splits your data into a training set and a testing set.
from sklearn.preprocessing import StandardScaler #scales your features so they all sit on a similar range
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score #check how good your model is after training.

# 1. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Scale features (logistic regression is sensitive to feature scale
# because it uses gradient-based optimization)
scaler = StandardScaler() #Creates the scaler object. It hasn't done anything yet.
X_train_scaled = scaler.fit_transform(X_train)
#fit means the scaler looks at your training data and learns the mean and standard
#deviation of each feature. transform means it then actually rescales the data using what it learned. fit_transform does both in one step.
X_test_scaled = scaler.transform(X_test) # use only transform on testdata

# 3. Fit the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 4. Predict
y_pred = model.predict(X_test_scaled) #Gives you hard class predictions, 0 or 1, for the test set
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # probability of class 1

# 5. Evaluate
print(accuracy_score(y_test, y_pred)) #Tells you what fraction of predictions were correct.
print(classification_report(y_test, y_pred)) #Gives you precision, recall, and F1 score for each class, not just overall accuracy.
print(roc_auc_score(y_test, y_proba)) #Measures how well the model separates the two classes across all possible thresholds.
#uses probabi;ities , no the hard predictions,which is why we pass y_proba here instead of y_pred.

NameError: name 'X' is not defined